# Procesos y Concurrencia en Elixir

Elixir se basa en el modelo de actores de la máquina virtual Erlang (BEAM). La concurrencia en Elixir no se trata de hilos del sistema operativo que comparten memoria, sino de miles de procesos aislados que se comunican mediante el paso de mensajes.

## 1. Procesos Básicos

Los procesos de Elixir son extremadamente ligeros. Puedes ejecutar cientos de miles en una sola máquina sin agotar los recursos.

In [ ]:
# Obtener el PID (Process Identifier) del proceso actual
pid_actual = self()
IO.inspect(pid_actual)

# Crear un nuevo proceso (spawn)
pid_hijo = spawn(fn -> 
  IO.puts("Hola desde un proceso nuevo!") 
end)

IO.inspect(pid_hijo)

# ¿Está vivo el proceso?
Process.alive?(pid_hijo)

## 2. Paso de Mensajes

Los procesos se comunican enviando mensajes asíncronos. Cada proceso tiene un "buzón" (mailbox) donde se almacenan los mensajes entrantes.

In [ ]:
send(self(), {:hola, "mundo"})

receive do
  {:hola, msg} -> "Recibido: #{msg}"
  {:error, _} -> "Hubo un error"
after
  5_000 -> "No llegó nada en 5 segundos"
end

## 3. Enlaces (Links) y Monitores

En Elixir, adoptamos la filosofía "Let it crash". Los procesos se organizan para que, si uno falla, otros puedan detectarlo y reaccionar.

- **Links**: Si un proceso falla, el otro también (bi-direccional).
- **Monitors**: Un proceso observa a otro (uni-direccional). Recibe un mensaje si el observado falla.

In [ ]:
# Spawn link: Crea un proceso y lo enlaza al actual
# spawn_link(fn -> raise "Oops!" end)

# Monitorear un proceso
pid = spawn(fn -> :ok end)
ref = Process.monitor(pid)

receive do
  {:DOWN, ^ref, :process, ^pid, razon} -> 
    IO.puts("El proceso #{inspect(pid)} murió por: #{inspect(razon)}")
end

## 4. Agentes (State Management)

Los `Agents` son abstracciones sencillas sobre los procesos para mantener un estado mutable.

In [ ]:
{:ok, agente} = Agent.start_link(fn -> [] end)

Agent.update(agente, fn lista -> ["manzana" | lista] end)
Agent.update(agente, fn lista -> ["pera" | lista] end)

estado = Agent.get(agente, fn lista -> lista end)
IO.inspect(estado) # ["pera", "manzana"]

Agent.stop(agente)

## 5. Tareas (Tasks)

Las `Tasks` se utilizan para ejecutar operaciones en segundo plano y, opcionalmente, recuperar su valor.

In [ ]:
tarea = Task.async(fn -> 
  Process.sleep(1000)
  "Resultado pesado"
end)

IO.puts("Haciendo otras cosas...")

resultado = Task.await(tarea)
IO.puts(resultado)

## 6. Introducción a OTP: GenServer y Supervisores

OTP (Open Telecom Platform) es un conjunto de librerías para construir sistemas tolerantes a fallos.

### GenServer
Es un comportamiento que implementa el patrón cliente-servidor de forma genérica.

### Supervisores
Son procesos que observan a otros procesos y los reinician si fallan, permitiendo la auto-sanación del sistema.

In [ ]:
defmodule MiServidor do
  use GenServer

  # API Cliente
  def start_link(estado_inicial) do
    GenServer.start_link(__MODULE__, estado_inicial)
  end

  def obtener_estado(pid) do
    GenServer.call(pid, :get_state)
  end

  # Callbacks Servidor
  @impl true
  def init(estado) do
    {:ok, estado}
  end

  @impl true
  def handle_call(:get_state, _from, estado) do
    {:reply, estado, estado}
  end
end

{:ok, pid} = MiServidor.start_link("Dato guardado")
MiServidor.obtener_estado(pid)